# Fine-Tuning Qwen/Qwen2.5-0.5B-Instruct for Dashboard Design

This notebook fine-tunes a small language model with **LoRA** so it can generate
structured JSON dashboard design recommendations from a short text description.

**Steps:**
1. Check GPU
2. Install libraries
3. Generate synthetic training data
4. Split into train / validation
5. Load model and tokenizer
6. Configure LoRA
7. Train with SFTTrainer
8. Save adapter
9. Run inference
10. Download adapter (optional)


## 1. Check GPU

Make sure you have selected a GPU runtime:
**Runtime → Change runtime type → T4 GPU**


In [ ]:
!nvidia-smi

## 2. Install Libraries

This installs all required packages. It takes about 1–2 minutes.


In [ ]:
!pip install -q -U transformers datasets peft trl accelerate bitsandbytes sentencepiece
print('All packages installed.')

## 3. Imports


In [ ]:
import json
import os
import random
import torch

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')


## 4. Generate Synthetic Training Data

We create 80 examples across 10 domains (8 variants each).
Each example has a dashboard brief as input and a structured JSON recommendation as output.


In [ ]:
# ── Domain definitions ────────────────────────────────────────────────────
DOMAINS = [
    'Sales', 'Finance', 'HR', 'Marketing', 'Logistics',
    'Healthcare', 'Energy Consumption', 'Customer Support',
    'E-Commerce', 'Project Management',
]

KPI_BY_DOMAIN = {
    'Sales':              ['Total Revenue','Monthly Sales Growth','Win Rate','Average Deal Size','Sales by Region','Pipeline Value','Quota Attainment','Churn Rate'],
    'Finance':            ['Net Profit Margin','Operating Expenses','Cash Flow','Budget Variance','Revenue vs. Cost','EBITDA','Accounts Receivable','Debt-to-Equity Ratio'],
    'HR':                 ['Headcount','Employee Turnover Rate','Time to Hire','Absenteeism Rate','Training Completion Rate','Employee Satisfaction Score','Diversity Ratio','Overtime Hours'],
    'Marketing':          ['Campaign ROI','Lead Conversion Rate','Cost per Lead','Website Traffic','Email Open Rate','Social Media Engagement','Customer Acquisition Cost','Brand Awareness Score'],
    'Logistics':          ['On-Time Delivery Rate','Shipment Volume','Warehouse Utilization','Order Fulfillment Time','Return Rate','Freight Cost per Unit','Inventory Turnover','Carrier Performance'],
    'Healthcare':         ['Patient Admission Rate','Average Length of Stay','Bed Occupancy Rate','Readmission Rate','Treatment Success Rate','Staff-to-Patient Ratio','Appointment No-Show Rate','Medication Error Rate'],
    'Energy Consumption': ['Total Energy Usage (kWh)','Peak Demand','Renewable Energy Share','Energy Cost per Unit','Carbon Emissions','Energy Intensity','Equipment Efficiency','Downtime Hours'],
    'Customer Support':   ['First Response Time','Ticket Resolution Rate','Customer Satisfaction (CSAT)','Average Handle Time','Escalation Rate','Open Tickets','Agent Utilization','Net Promoter Score (NPS)'],
    'E-Commerce':         ['Gross Merchandise Value (GMV)','Conversion Rate','Cart Abandonment Rate','Average Order Value','Return Rate','Customer Lifetime Value','Revenue by Category','Traffic by Channel'],
    'Project Management': ['Task Completion Rate','Budget Utilization','Schedule Variance','Resource Allocation','Risk Count','Milestone Achievement Rate','Team Velocity','Defect Rate'],
}

USER_GOALS_BY_DOMAIN = {
    'Sales':              ['track revenue trends','compare regional performance','monitor pipeline health','identify top-performing products','forecast quarterly targets','analyze win/loss ratios','evaluate rep performance','detect churn risk'],
    'Finance':            ['monitor cash flow','compare budget vs. actuals','track profit margins','analyze expense categories','forecast revenue','evaluate financial health','identify cost overruns','review debt levels'],
    'HR':                 ['track headcount changes','monitor turnover trends','evaluate hiring efficiency','analyze absenteeism patterns','review training progress','assess workforce diversity','identify overtime hotspots','measure employee satisfaction'],
    'Marketing':          ['evaluate campaign performance','track lead funnel','compare channel ROI','monitor website traffic','analyze email engagement','measure brand reach','optimize ad spend','identify top-converting campaigns'],
    'Logistics':          ['monitor delivery performance','track shipment volumes','optimize warehouse usage','analyze fulfillment times','review return rates','compare carrier performance','manage inventory levels','reduce freight costs'],
    'Healthcare':         ['monitor patient admissions','track bed occupancy','analyze readmission patterns','evaluate treatment outcomes','manage staff ratios','reduce no-show rates','track medication errors','optimize length of stay'],
    'Energy Consumption': ['monitor total energy usage','identify peak demand periods','track renewable share','analyze energy costs','reduce carbon emissions','improve equipment efficiency','detect downtime patterns','benchmark energy intensity'],
    'Customer Support':   ['track response times','monitor ticket resolution','measure customer satisfaction','analyze handle time trends','reduce escalation rates','manage open ticket backlog','optimize agent workload','improve NPS scores'],
    'E-Commerce':         ['track sales performance','analyze conversion funnel','reduce cart abandonment','monitor order values','manage return rates','evaluate customer lifetime value','compare category revenue','analyze traffic sources'],
    'Project Management': ['track task completion','monitor budget utilization','identify schedule delays','manage resource allocation','track risk exposure','review milestone progress','measure team velocity','analyze defect trends'],
}

DATA_FIELDS_BY_DOMAIN = {
    'Sales':              ['date','region','product','rep_name','revenue','deals_closed','pipeline_stage'],
    'Finance':            ['date','department','account','budget','actual_spend','profit','expense_category'],
    'HR':                 ['date','department','employee_id','hire_date','exit_date','training_status','satisfaction_score'],
    'Marketing':          ['date','channel','campaign_name','impressions','clicks','leads','conversions','spend'],
    'Logistics':          ['date','carrier','region','shipment_id','delivery_status','fulfillment_time','freight_cost'],
    'Healthcare':         ['date','ward','patient_id','admission_type','length_of_stay','readmitted','staff_count'],
    'Energy Consumption': ['date','facility','meter_id','kwh_used','peak_demand','renewable_kwh','cost'],
    'Customer Support':   ['date','agent_id','ticket_id','channel','resolution_time','csat_score','escalated'],
    'E-Commerce':         ['date','product_category','sku','orders','revenue','returns','traffic_source','cart_events'],
    'Project Management': ['date','project_id','task_id','assignee','status','planned_hours','actual_hours','risk_level'],
}

TASK_TO_CHART = [
    ('trend over time',               'line chart'),
    ('category comparison',           'bar chart'),
    ('composition/share',             'stacked bar chart or donut chart'),
    ('relationship/correlation',      'scatter plot'),
    ('detailed records',              'table'),
    ('main KPIs',                     'KPI cards'),
    ('patterns over time/categories', 'heatmap'),
]

LAYOUT_OPTIONS = [
    'Top row: KPI cards. Middle: main chart (full width). Bottom: supporting charts side by side.',
    'Left panel: KPI cards (vertical). Right panel: main chart on top, detail table below.',
    'Top: KPI cards row. Center: two charts side by side. Bottom: trend line chart full width.',
    'Header: KPI summary cards. Body: tabbed views per category. Footer: data table.',
    'Single-page layout: KPI cards at top, heatmap in center, bar chart and line chart at bottom.',
]

INTERACTION_OPTIONS = [
    'Date range filter','Region/department dropdown filter','Drill-down on chart click',
    'Tooltip on hover','Export to CSV button','Search/filter on table',
    'Toggle between chart types','Highlight on selection','Cross-filter between charts','Zoom on time axis',
]

COLOR_OPTIONS = [
    'Blue-green palette for positive trends; red for negative deviations. Clear axis labels with units.',
    'Neutral grey base with accent colors per category. Consistent legend placement.',
    'Traffic-light colors (green/yellow/red) for KPI status indicators. Accessible contrast ratios.',
    'Sequential color scale for heatmap. Diverging palette for variance charts.',
    'Brand-aligned primary color for main metrics; muted tones for secondary data.',
]

RATIONALE_SNIPPETS = [
    'Line charts effectively communicate trends over continuous time periods.',
    'Bar charts allow quick comparison across discrete categories.',
    'KPI cards provide immediate visibility into the most critical metrics.',
    'Heatmaps reveal patterns across two dimensions simultaneously.',
    'Donut/stacked bar charts show part-to-whole relationships clearly.',
    'Tables support detailed record inspection and sorting.',
    'Scatter plots expose correlations and outliers between two variables.',
    'Drill-down interactions let users move from summary to detail without leaving the dashboard.',
    'Consistent color coding reduces cognitive load and speeds interpretation.',
    'Filtering controls empower users to focus on relevant subsets of data.',
]

INSTRUCTION = 'Generate a structured dashboard design recommendation based on the given dashboard brief.'

def build_example(domain, variant_index):
    rng = random.Random(hash(domain) + variant_index * 31)
    kpis   = rng.sample(KPI_BY_DOMAIN[domain], k=3)
    goals  = rng.sample(USER_GOALS_BY_DOMAIN[domain], k=3)
    fields = rng.sample(DATA_FIELDS_BY_DOMAIN[domain], k=5)
    pairs  = rng.sample(TASK_TO_CHART, k=3)
    kpi_map = []
    for i, kpi in enumerate(kpis):
        task, chart = pairs[i]
        kpi_map.append({'kpi': kpi, 'user_task': task, 'recommended_chart': chart, 'rationale': rng.choice(RATIONALE_SNIPPETS)})
    return {
        'instruction': INSTRUCTION,
        'input': f'Domain: {domain}\nAvailable data fields: {", ".join(fields)}\nKey metrics to track: {", ".join(kpis)}\nUser goals: {", ".join(goals)}',
        'output': {
            'context_summary': f'This {domain} dashboard helps users {goals[0]} and {goals[1]}. It focuses on {", ".join(kpis)}.',
            'kpi_task_chart_mapping': kpi_map,
            'layout_hierarchy': rng.choice(LAYOUT_OPTIONS),
            'labels_scales_colors': rng.choice(COLOR_OPTIONS),
            'interactions': rng.sample(INTERACTION_OPTIONS, k=3),
            'design_rationales': rng.sample(RATIONALE_SNIPPETS, k=2),
        },
    }

examples = [build_example(d, v) for d in DOMAINS for v in range(8)]
print(f'Generated {len(examples)} examples across {len(DOMAINS)} domains.')


## 5. Split into Train / Validation

90% train, 10% validation. We format each example into the instruction-tuning text format.


In [ ]:
SYSTEM_PROMPT = (
    'You are a dashboard design assistant. '
    'You generate structured, practical, and heuristic-aligned dashboard design recommendations.'
)

def format_example(ex):
    return (
        f'<|system|>\n{SYSTEM_PROMPT}\n'
        f'<|user|>\nInstruction:\n{ex["instruction"]}\n\nDashboard brief:\n{ex["input"]}\n'
        f'<|assistant|>\n{json.dumps(ex["output"], indent=2, ensure_ascii=False)}'
    )

random.seed(42)
random.shuffle(examples)
split = int(len(examples) * 0.9)
train_data = [{'text': format_example(e)} for e in examples[:split]]
val_data   = [{'text': format_example(e)} for e in examples[split:]]

print(f'Train examples      : {len(train_data)}')
print(f'Validation examples : {len(val_data)}')
print('\n--- Sample training text (first 500 chars) ---')
print(train_data[0]['text'][:500])


## 6. Load Dataset with Hugging Face datasets


In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_list(train_data)
val_dataset   = Dataset.from_list(val_data)

print(train_dataset)
print(val_dataset)


## 7. Load Model and Tokenizer

We load `Qwen/Qwen2.5-0.5B-Instruct`. The download is about 1 GB and is cached after the first run.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'

print(f'Loading tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Loading model: {MODEL_NAME}')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.config.use_cache = False
print('Model loaded.')


## 8. Configure LoRA

LoRA (Low-Rank Adaptation) adds a small number of trainable parameters on top of the frozen base model.
This makes fine-tuning fast and memory-efficient.

- `r=8` — rank (controls how many parameters are trained)
- `lora_alpha=16` — scaling factor
- `target_modules` — which layers to adapt


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj'],
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 9. Train with SFTTrainer

We use Hugging Face TRL's `SFTTrainer` which handles the instruction-tuning format automatically.

Settings:
- 1 epoch (fast, good for testing)
- batch size 1
- gradient accumulation 4 (effective batch size = 4)
- learning rate 2e-4
- max sequence length 1024 tokens


In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = '/content/qwen-dashboard-lora'

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    save_steps=25,
    save_total_limit=1,
    eval_strategy='epoch',
    fp16=True,
    report_to='none',
    dataloader_num_workers=0,
    max_length=1024,
    dataset_text_field='text',
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=sft_config,
)

print('Starting training ...')
trainer.train()
print('Training complete.')

## 10. Save the LoRA Adapter

We save only the LoRA adapter weights (not the full model). This is much smaller (~10–50 MB).


In [ ]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Adapter saved to: {OUTPUT_DIR}')
!ls -lh {OUTPUT_DIR}


## 11. Run Inference with the Fine-Tuned Model

We test the fine-tuned model on a fixed dashboard brief and display the structured JSON output.


In [ ]:
from peft import PeftModel

TEST_BRIEF = (
    'Create a dashboard for a sales manager who wants to monitor monthly revenue, '
    'sales by region, top products, conversion rate, and customer churn. '
    'The data contains date, region, product category, revenue, number of leads, '
    'conversions, and churn status.'
)

SCHEMA_HINT = json.dumps({
    'context_summary': '',
    'kpi_task_chart_mapping': [],
    'layout_hierarchy': '',
    'labels_scales_colors': '',
    'interactions': [],
    'design_rationales': [],
}, indent=2)

messages = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': f'Dashboard brief:\n{TEST_BRIEF}\n\nPlease fill in the following JSON schema with a complete dashboard design recommendation. Return only valid JSON, no extra text.\n\n{SCHEMA_HINT}'},
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors='pt').to(model.device)

print('Running inference ...')
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=1024,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=tokenizer.eos_token_id,
    )

generated_ids = output_ids[0][inputs['input_ids'].shape[1]:]
response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
print('\n--- Model Response ---')
print(response)


## 12. Parse and Display the Result


In [ ]:
import re

def try_parse_json(text):
    cleaned = text.strip()
    # Strip markdown code fences
    if cleaned.startswith('```'):
        lines = cleaned.splitlines()
        cleaned = '\n'.join(lines[1:-1]) if len(lines) > 2 else cleaned
    # Extract first JSON object if there is surrounding text
    match = re.search(r'\{.*\}', cleaned, re.DOTALL)
    if match:
        cleaned = match.group(0)
    # Remove trailing commas before ] or }
    cleaned = re.sub(r',\s*([}\]])', r'\1', cleaned)
    try:
        return json.loads(cleaned), None
    except json.JSONDecodeError as e:
        return None, str(e)

parsed, error = try_parse_json(response)
if error:
    print(f'WARNING: Output is not valid JSON.\nError: {error}')
    print('\n--- Raw output ---')
    print(response)
else:
    print('Valid JSON output:')
    print(json.dumps(parsed, indent=2, ensure_ascii=False))

## 13. Download the Adapter (Optional)

Run this cell to download the LoRA adapter as a ZIP file to your computer.
You can then use it locally with `scripts/06_inference_finetuned_model.py`.


In [ ]:
import shutil
from google.colab import files

zip_path = '/content/qwen-dashboard-lora.zip'
shutil.make_archive('/content/qwen-dashboard-lora', 'zip', OUTPUT_DIR)
print(f'ZIP created: {zip_path}')
files.download(zip_path)
